# 🛡️ SentiLLM - Security Vulnerability Analysis Model Training

## Fine-tuning Llama-3-8B with Unsloth + QLoRA + NEFTune

**Training Stack:**
- 🦥 **Unsloth**: 2x faster training with 70% less memory
- 🔧 **QLoRA**: 4-bit quantization with LoRA adapters
- 🎯 **NEFTune**: Noise Embedding for better generalization
- 📊 **Weights & Biases**: Experiment tracking
- 🦙 **Llama-3-8B**: Base model

**Dataset:** Synthetic CVE vulnerability analysis data (Alpaca format)

## 1. Install Dependencies

Install Unsloth and required packages. Run this cell first!

In [ ]:
# ============================================================
# RUNPOD - UNSLOTH OFFICIAL DOCKER IMAGE
# Image: unslothai/unsloth:latest
# ============================================================
# Unsloth is pre-installed! Only need extra training packages.

import os
# Disable Hopper TMA features for Ampere GPUs (RTX 3090)
os.environ["UNSLOTH_USE_TRITON"] = "1"  # Force Triton kernels
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# Step 1: Check environment
print("📋 Environment Check:")
import torch
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.version.cuda}")
print(f"   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

# Step 2: Verify Unsloth is installed
try:
    import unsloth
    print(f"   Unsloth: ✅ Pre-installed")
except ImportError:
    print("   Unsloth: ❌ Not found - wrong image?")

print("\n" + "="*60)
print("🔧 INSTALLING ADDITIONAL PACKAGES...")
print("="*60 + "\n")

# Step 3: Install training dependencies (including psutil for SFTTrainer)
!pip install -q wandb datasets sentencepiece psutil

print("\n" + "="*60)
print("✅ READY TO TRAIN!")
print("="*60)
print("\n🚀 Run the next cell to continue")
print("="*60)

## 2. Initialize Weights & Biases

Login to W&B for experiment tracking and visualization.

In [ ]:
import wandb
import os

# Fix notebook name detection warning
os.environ["WANDB_NOTEBOOK_NAME"] = "SentiLLM_Training"

# Initialize Weights & Biases
wandb.login()

# RTX 3090 Optimized Configuration (24GB VRAM)
wandb.init(
    project="SentiLLM-CVE-Analyzer",
    name="llama3-8b-qlora-neftune-rtx3090",
    tags=["llama3", "qlora", "neftune", "security", "cve", "rtx3090"],
    config={
        "gpu": "RTX 3090 (24GB)",
        "model": "unsloth/llama-3-8b-bnb-4bit",
        "lora_r": 32,
        "lora_alpha": 32,
        "neftune_noise_alpha": 5,
        "epochs": 3,
        "batch_size": 4,
        "gradient_accumulation": 4,
        "effective_batch_size": 16,
        "learning_rate": 2e-4,
        "max_seq_length": 4096,
    }
)

print("✅ Weights & Biases initialized!")
print("🖥️  Optimized for: NVIDIA RTX 3090 (24GB VRAM)")

## 3. Load Llama-3-8B with Unsloth (4-bit Quantization)

Using Unsloth's optimized loader for 2x faster training and 70% less VRAM.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Configuration
max_seq_length = 4096  # Supports longer CVE reports
dtype = None  # Auto-detect (Float16 for Tesla T4, BFloat16 for Ampere+)
load_in_4bit = True  # Use 4-bit quantization for memory efficiency

# Load Llama-3-8B with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",  # Pre-quantized for speed
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    # token="hf_...",  # Add your HuggingFace token if needed
)

print(f"✅ Model loaded successfully!")
print(f"📊 Max sequence length: {max_seq_length}")
print(f"🔧 4-bit quantization: {load_in_4bit}")

## 4. Configure QLoRA Adapters

Apply LoRA adapters to specific layers for efficient fine-tuning.

In [ ]:
# Configure LoRA adapters - RTX 3090 Optimized
# Higher rank for better quality (24GB VRAM allows this)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # Increased LoRA rank for RTX 3090 - better quality
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj",     # MLP layers
    ],
    lora_alpha=32,  # Scaling factor (typically equals r)
    lora_dropout=0,  # No dropout for deterministic training (Unsloth optimized)
    bias="none",     # No bias training
    use_gradient_checkpointing="unsloth",  # Long context support, saves VRAM
    random_state=42,
    use_rslora=False,  # Rank-stabilized LoRA (optional)
    loftq_config=None, # LoftQ quantization (optional)
)

# Print trainable parameters
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"🎯 Trainable params: {trainable_params:,} ({100 * trainable_params / all_param:.2f}%)")
    print(f"📊 Total params: {all_param:,}")

print_trainable_parameters(model)
print(f"✅ LoRA configured with r=32, alpha=32 (RTX 3090 optimized)")

## 5. Load and Prepare Training Dataset

Load the synthetic CVE analysis dataset and format it for Alpaca-style training with EOS token.

In [ ]:
from datasets import load_dataset, Dataset
import json

# Load training data from JSON file
DATA_PATH = "./training_data_detailed.json"  # Updated path

# Load dataset
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Convert to HuggingFace Dataset
dataset = Dataset.from_list(raw_data)

print(f"✅ Dataset loaded: {len(dataset)} samples")
print(f"\n📋 Dataset columns: {dataset.column_names}")
print(f"\n📋 Sample entry:")
print(f"Scenario: {dataset[0].get('scenario', 'N/A')}")
print(f"Language: {dataset[0].get('language', 'N/A')}")
print(f"Is Vulnerable: {dataset[0].get('is_vulnerable', 'N/A')}")
print(f"Instruction: {dataset[0]['instruction'][:80]}...")
print(f"Input length: {len(dataset[0]['input'])} chars")
print(f"Output length: {len(dataset[0]['output'])} chars")

## 6. Define Prompt Template with EOS Token

Create the Alpaca-style prompt template with proper EOS token for training termination.

In [ ]:
# Enhanced Alpaca-style prompt template with metadata
# Includes scenario, language, and vulnerability status for better learning

EOS_TOKEN = tokenizer.eos_token  # Get the model's EOS token

# Enhanced prompt template with metadata context
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Metadata:
- Scenario: {scenario}
- Language: {language}
- Expected Result: {expected_result}

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

def formatting_prompts_func(examples):
    """Format dataset examples into enhanced Alpaca prompt format with metadata"""
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    
    # Get metadata fields (with safe defaults for missing/empty values)
    scenarios = examples.get("scenario", [])
    languages = examples.get("language", [])
    is_vulnerable = examples.get("is_vulnerable", [])
    
    texts = []
    for i, (instruction, input_text, output) in enumerate(zip(instructions, inputs, outputs)):
        # Safe get with fallback for None/empty values
        scenario = scenarios[i] if i < len(scenarios) and scenarios[i] else "general"
        language = languages[i] if i < len(languages) and languages[i] else "mixed"
        
        # Determine expected result from is_vulnerable (handle None/missing)
        vuln_status = is_vulnerable[i] if i < len(is_vulnerable) else None
        if vuln_status is True:
            expected = "VULNERABLE"
        elif vuln_status is False:
            expected = "NOT VULNERABLE"
        else:
            expected = "ANALYZE"  # For None or missing values
        
        # Format with template and append EOS token
        text = alpaca_prompt.format(
            scenario=scenario,
            language=language,
            expected_result=expected,
            instruction=instruction,
            input=input_text,
            output=output
        ) + EOS_TOKEN
        texts.append(text)
    
    return {"text": texts}

# Apply formatting to dataset
dataset = dataset.map(formatting_prompts_func, batched=True)

# Count metadata statistics
print(f"✅ Dataset formatted with metadata + EOS token: {EOS_TOKEN}")
print(f"\n📊 Metadata Statistics:")
try:
    scenarios = [s for s in dataset["scenario"] if s]
    languages = [l for l in dataset["language"] if l]
    vulns = [v for v in dataset["is_vulnerable"] if v is not None]
    print(f"   - Samples with scenario: {len(scenarios)}/{len(dataset)}")
    print(f"   - Samples with language: {len(languages)}/{len(dataset)}")
    print(f"   - Samples with is_vulnerable: {len(vulns)}/{len(dataset)}")
except:
    print("   - (Metadata columns may vary)")

print(f"\n📝 Sample formatted prompt (first 700 chars):")
print(dataset[0]["text"][:700])
print(f"\n... (truncated)")

## 7. Configure Training with NEFTune

Set up the SFTTrainer with NEFTune noise embedding for better generalization.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

# ============================================================
# RTX 3090 OPTIMIZED TRAINING CONFIGURATION (24GB VRAM)
# Using SFTConfig instead of TrainingArguments
# ============================================================

sft_config = SFTConfig(
    # Output settings
    output_dir="./outputs",
    
    # Training hyperparameters - RTX 3090 optimized
    num_train_epochs=3,
    per_device_train_batch_size=4,  # Increased for RTX 3090 (24GB)
    gradient_accumulation_steps=4,   # Effective batch size = 4 * 4 = 16
    
    # Optimizer settings
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,  # 3% warmup
    
    # Precision - RTX 3090 (Ampere) supports BF16
    fp16=False,  # Disable FP16
    bf16=True,   # RTX 3090 has native BF16 support
    
    # Logging (Weights & Biases integration)
    logging_steps=5,  # More frequent logging
    logging_strategy="steps",
    report_to="wandb",
    
    # Saving
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    
    # Optimization - RTX 3090 specific
    optim="adamw_8bit",  # Memory-efficient optimizer
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,  # Gradient clipping
    
    # Performance
    dataloader_num_workers=4,  # Multi-threaded data loading
    dataloader_pin_memory=True,  # Faster GPU transfer
    
    # SFTConfig specific
    max_seq_length=max_seq_length,
    dataset_text_field="text",
    packing=True,  # Enable packing for faster training
    dataset_num_proc=4,  # Parallel data processing
    
    # NEFTune Configuration
    neftune_noise_alpha=5,
    
    # Misc
    seed=42,
    data_seed=42,
    group_by_length=True,  # Group similar length samples for efficiency
)

# Initialize SFTTrainer with SFTConfig
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)

print("="*60)
print("🖥️  RTX 3090 OPTIMIZED TRAINING CONFIGURATION")
print("="*60)
print(f"📊 Training settings:")
print(f"   - GPU: NVIDIA RTX 3090 (24GB VRAM)")
print(f"   - Precision: BF16 (native Ampere support)")
print(f"   - Epochs: {sft_config.num_train_epochs}")
print(f"   - Batch size: {sft_config.per_device_train_batch_size}")
print(f"   - Gradient accumulation: {sft_config.gradient_accumulation_steps}")
print(f"   - Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"   - Learning rate: {sft_config.learning_rate}")
print(f"   - LoRA rank: 32 (high quality)")
print(f"   - NEFTune noise alpha: {sft_config.neftune_noise_alpha}")
print(f"   - Packing: Enabled (faster training)")
print(f"   - Data workers: 4")
print("="*60)

## 8. Check GPU Memory Before Training

Verify GPU memory usage before starting training.

In [ ]:
# Check GPU memory usage
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"🖥️  GPU: {gpu_stats.name}")
print(f"📊 Max memory: {max_memory} GB")
print(f"💾 Reserved memory: {start_gpu_memory} GB")
print(f"✅ Available for training: {max_memory - start_gpu_memory:.2f} GB")

## 9. 🚀 Start Training!

Run the fine-tuning process. This will take a while depending on your GPU.

In [ ]:
# Start training!
print("🚀 Starting training...")
print("="*60)

trainer_stats = trainer.train()

print("="*60)
print("✅ Training complete!")

## 10. Training Statistics

View detailed training metrics and memory usage.

In [ ]:
# Calculate training statistics
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print("="*60)
print("📊 TRAINING STATISTICS")
print("="*60)
print(f"⏱️  Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"📈 Samples/second: {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"🔢 Steps/second: {trainer_stats.metrics['train_steps_per_second']:.2f}")
print(f"📉 Final loss: {trainer_stats.metrics['train_loss']:.4f}")
print()
print("💾 MEMORY USAGE")
print(f"   Peak reserved memory: {used_memory} GB")
print(f"   Memory for LoRA training: {used_memory_for_lora} GB")
print(f"   Memory usage: {used_percentage}%")
print("="*60)

# Log final metrics to W&B
wandb.log({
    "final_loss": trainer_stats.metrics['train_loss'],
    "training_time_seconds": trainer_stats.metrics['train_runtime'],
    "peak_memory_gb": used_memory,
    "lora_memory_gb": used_memory_for_lora,
})

## 11. Test the Fine-tuned Model

Run inference to verify the model works correctly on CVE analysis tasks.

In [ ]:
# Enable inference mode (faster generation)
FastLanguageModel.for_inference(model)

# Test prompt - simulating a real CVE analysis request
test_instruction = """You are a Senior Product Security Engineer. Analyze the provided 'CVE Intelligence Report' and the 'Dependency File'. 
1. Determine if the project is vulnerable based on Semantic Versioning rules.
2. Ignore comments and unrelated libraries in the file.
3. Provide a strictly valid JSON response containing the risk assessment, reasoning, and remediation command."""

test_input = """--- CVE INTELLIGENCE REPORT ---
CVE ID: CVE-2024-12345
Title: Remote Code Execution (RCE) in 'requests' library
Severity: CRITICAL (CVSS 9.8)
Affected Versions: requests < 2.31.0
Description: A vulnerability in the 'requests' library allows attackers to execute arbitrary code on the target system. All users running affected versions should upgrade immediately.

--- TARGET DEPENDENCY FILE (requirements.txt) ---
# Project Dependencies
# Core dependencies
flask==2.3.0
pandas==2.0.0

# Networking utilities
requests==2.28.1  # TODO: Update this later
urllib3==1.26.15

# Auth & Security
PyJWT==2.8.0
"""

# ============================================================
# INFERENCE FORMAT - Model KENDİ KARAR VERECEK!
# ============================================================
# Training'de: expected_result = "VULNERABLE" veya "NOT VULNERABLE" 
#              → Model bu pattern'i öğrendi
# Inference'da: expected_result = "ANALYZE"
#              → Model kendi analiz edip karar verecek
# ============================================================

test_prompt = alpaca_prompt.format(
    scenario="vulnerability_analysis",  # Senaryo tipi
    language="Python",                   # Dosya dili
    expected_result="ANALYZE",           # ⚠️ Model kendi karar verecek!
    instruction=test_instruction,
    input=test_input,
    output=""  # Boş bırak - model dolduracak
)

# Tokenize
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

# Generate response
print("🔍 Testing model inference...")
print("="*60)
print("📝 Expected Result: ANALYZE (model kendi karar verecek)")
print("🎯 Doğru cevap: VULNERABLE (requests 2.28.1 < 2.31.0)")
print("="*60)

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.1,  # Low temperature for deterministic output
    top_p=0.9,
    do_sample=True,
    use_cache=True,
)

# Decode and print response
response = tokenizer.batch_decode(outputs)[0]

# Extract only the response part
response_start = response.find("### Response:") + len("### Response:")
response_text = response[response_start:].strip()

# Remove EOS token from display
response_text = response_text.replace(EOS_TOKEN, "")

print("\n📋 MODEL RESPONSE:")
print("="*60)
print(response_text)
print("="*60)

# Quick validation
if "VULNERABLE" in response_text.upper() or '"is_vulnerable": true' in response_text.lower():
    print("\n✅ Model doğru karar verdi: VULNERABLE")
elif "NOT VULNERABLE" in response_text.upper() or '"is_vulnerable": false' in response_text.lower():
    print("\n❌ Model yanlış karar verdi: NOT VULNERABLE (olmalıydı: VULNERABLE)")
else:
    print("\n⚠️ Model kararı net değil - response'u kontrol et")

## 12. Save the Model

Save the fine-tuned LoRA adapters and optionally merge with base model.

In [ ]:
# Save paths
LORA_OUTPUT_DIR = "./sentillm-lora"
MERGED_OUTPUT_DIR = "./sentillm-merged"

# Option 1: Save only LoRA adapters (small, ~50MB)
# This is recommended for deployment with the base model
print("💾 Saving LoRA adapters...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)
print(f"✅ LoRA adapters saved to: {LORA_OUTPUT_DIR}")

# Option 2: Save merged model (larger, full model)
# Uncomment if you want a standalone model
# print("💾 Saving merged model (16-bit)...")
# model.save_pretrained_merged(MERGED_OUTPUT_DIR, tokenizer, save_method="merged_16bit")
# print(f"✅ Merged model saved to: {MERGED_OUTPUT_DIR}")

## 13. Push to Hugging Face Hub (Optional)

Upload your fine-tuned model to Hugging Face for easy sharing and deployment.

In [ ]:
# ============================================================
# HUGGING FACE HUB'A MODEL YÜKLEME
# ============================================================

# 1. Ayarları düzenle
HF_USERNAME = "YOUR_USERNAME"  # ← HuggingFace kullanıcı adın
HF_TOKEN = "hf_xxxxxxxxxxxx"   # ← Token'ını buraya yapıştır
MODEL_NAME = "sentillm-cve-analyzer"

# 2. Token ile giriş yap
from huggingface_hub import login
login(token=HF_TOKEN)
print("✅ HuggingFace'e giriş yapıldı!")

# 3. LoRA adaptörlerini yükle (küçük, ~50MB)
print(f"\n📤 Uploading to: {HF_USERNAME}/{MODEL_NAME}-lora")
model.push_to_hub(
    f"{HF_USERNAME}/{MODEL_NAME}-lora",
    token=HF_TOKEN,
    private=False,  # ✅ PUBLIC REPO
)
tokenizer.push_to_hub(
    f"{HF_USERNAME}/{MODEL_NAME}-lora",
    token=HF_TOKEN,
)
print(f"✅ LoRA adapters uploaded!")

# 4. (Opsiyonel) GGUF formatında yükle - llama.cpp için
# model.push_to_hub_gguf(
#     f"{HF_USERNAME}/{MODEL_NAME}-gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
#     token=HF_TOKEN,
# )

print(f"""
🎉 MODEL YÜKLEME TAMAMLANDI!

📦 Model: https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}-lora

🔧 Kullanım:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained("{HF_USERNAME}/{MODEL_NAME}-lora")
""")

## 14. Finish W&B Run

Close the Weights & Biases experiment run.

In [ ]:
# Finish W&B run
wandb.finish()

print("="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)
print(f"""
📁 Model saved to: {LORA_OUTPUT_DIR}

🚀 Next Steps:
1. Load the model for inference:
   from unsloth import FastLanguageModel
   model, tokenizer = FastLanguageModel.from_pretrained("{LORA_OUTPUT_DIR}")

2. View training logs at: https://wandb.ai

3. Deploy with vLLM or llama.cpp for production use
""")

---

## 📚 RTX 3090 Optimized Training Summary

| Component | Configuration |
|-----------|---------------|
| **GPU** | NVIDIA RTX 3090 (24GB VRAM) |
| **Base Model** | Llama-3-8B (4-bit quantized) |
| **Method** | QLoRA (r=32, alpha=32) |
| **Precision** | BF16 (native Ampere support) |
| **NEFTune** | noise_alpha=5 |
| **Optimizer** | AdamW 8-bit |
| **Learning Rate** | 2e-4 with cosine scheduler |
| **Batch Size** | 4 (effective: 16 with gradient accumulation) |
| **Epochs** | 3 |
| **Packing** | ✅ Enabled (faster training) |
| **EOS Token** | ✅ Added to all training samples |
| **Tracking** | Weights & Biases |

### RTX 3090 Optimizations:
- 🎮 **Larger Batch Size**: 4 (vs 2) - better GPU utilization
- 🔧 **Higher LoRA Rank**: 32 (vs 16) - better model quality
- ⚡ **BF16 Precision**: Native Ampere support for faster training
- 📦 **Packing Enabled**: More efficient sequence processing
- 🔄 **4 Data Workers**: Parallel data loading
- 💾 **~18-20GB VRAM Usage**: Optimal for 24GB card

### Expected Training Time:
- ~1000 samples: ~15-20 minutes
- ~2000 samples: ~30-40 minutes
- ~5000 samples: ~1-1.5 hours